# LSCI 220 Assignment 2
## Dominic Kelly, Id 198511370

# Introduction
A continuation of the processing done for Assignment 1.

# Beginning of material repeated from Assignment 1
Rip It Up was a New Zealand music magazine that was launched in 1977 and finally ceased publication in 2015. It was the closest thing that the New Zealand contemporary music scene had to a *New Musical Express* or *Rolling Stone*. In 1988, John Dix wrote in Stranded In Paradise (a history of New Zealand Rock 'n Roll) "New Zealand's best promotional outlet for what's happening out there in the real world of rock'n'roll is neither radio nor television, but *Rip It Up*, a monthly freebie that refuses to die."

On November 11, 2025, it was announced that the Papers Past website now holds more than 20 years of free, searchable and downloadable [content](https://paperspast.natlib.govt.nz/periodicals/rip-it-up) from Rip It Up for the years 1977-1998.

I downloaded all the album reviews for Rip It Up's first five months of publication (June - October 1977) and for the five months starting on its 20th anniversary.

I made sure that each review's metadata (artist, album title etc) was formatted consistently. Since Papers Past provides an image of the printed page as well as the generated text, I corrected any errors that I could tell had been created during the scanning process, but *not* grammatical or spelling errors that were apparent in the original.

In [26]:
# some preliminary imports, setting up static data etc
from ast import pattern
import nltk
import string
from nltk.corpus import PlaintextCorpusReader
from nltk import FreqDist
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import re

from enum import Enum

class Year(Enum):
    Y1977 = 0
    Y1997 = 1

punctuation_set = set(string.punctuation)

# additional punctuation are pieces of text that I found were appearing in the results 
# that I would like to have been removed with other punctuation
additional_punctuation = set(["'s", "'m", "'", '.', '"', "n't","'ve",
                              '’', '“', '”', '’;', '’’', '.)',
                              '–', '—', '...', '``', "''", '‘', 
                              '•', '”.', '",', '”,', '.):', '":', '’)',
                              '.”', ".'", '’,','’),', '–,', '—,', '’.', 
                              '(‘', '".', ').', '."', "')", '!!'])

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

nltk.download('wordnet')
nltk.download('omw-1.4') 

lemmatizer = WordNetLemmatizer()
corpus_root = '.'  # Directory containing text files
file_pattern = r'.*\.txt'  # Pattern to match .txt files
tokeniser = nltk.tokenize.WordPunctTokenizer()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\OEM\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\OEM\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\OEM\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


The ```do_scrubbing()``` function below does five types of pre-processing that I want to apply to the corpora. By default all are applied, which is how it is used below.

In [27]:
# define a few functions.

def do_scrubbing(tokens: list, remove_punctuation=True, remove_additional_puctuation=True, remove_stopwords=True, lowercase=True, lemmatize=True) -> list:
    # Most of the cleaning out of punctuation, stopwords, plural forms etc, all in one place.
    print("Initial token count:", len(tokens))
    if lowercase:
        tokens = [token.lower() for token in tokens]
    if (remove_punctuation):
        tokens = [word for word in tokens if word not in punctuation_set]
        # tokens = [token.translate(str.maketrans('', '', string.punctuation)) for token in tokens]

    if remove_additional_puctuation:
        tokens = [word for word in tokens if word not in additional_punctuation]

    if remove_stopwords:
        tokens = [word for word in tokens if word not in stop_words]

    if lemmatize:
        tokens = [lemmatizer.lemmatize(token) for token in tokens]
    print("Final token count after scrubbing:", len(tokens))
    return tokens

# for analysing the text of the reviews, I prefer to exclude the headers that contain album and artist names, record labels and reviewer names
def remove_header_lines(body: str, pattern: str) -> str:
    lines = body.split('\n')
    reg_exp = re.compile(pattern)
    cleaned_lines = [line for line in lines if not reg_exp.search(line)]
    return '\n'.join(cleaned_lines)

def draw_wordcloud(frequencies: FreqDist, title: str):
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(dict(frequencies.most_common(20)))
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off") # Turn off the axis labels
    plt.title(title)
    plt.show()

def normalise_frequencies(frequencies: FreqDist, token_count: int) -> FreqDist:
    normalized_frequencies = FreqDist()
    for word, freq in frequencies.items():
        norm_freq = freq / token_count * 100
        normalized_frequencies[word] = norm_freq
    return normalized_frequencies

In [28]:
# now the actual work

corpus = PlaintextCorpusReader(corpus_root, file_pattern)

raw_review_corpus = [None, None]
review_corpus = [None, None]
review_tokens = [None, None]
scrubbed_tokens = [None, None]
frequencies = [None, None]
normalised_frequencies = [None, None]

# Access the text

raw_review_corpus[Year.Y1977.value] = corpus.raw(['RECORDS 77.06.01.txt', 'RECORDS 77.07.01.txt', 'RECORDS 77.08.01.txt', 'RECORDS 77.09.01.txt', 'RECORDS 77.10.01.txt', 'RECORDS 77.11.01.txt', 'RECORDS 77.12.01.txt'])
raw_review_corpus[Year.Y1997.value] = corpus.raw(['ALBUMS 97.06.01.txt', 'ALBUMS 97.07.01.txt', 'ALBUMS 97.08.01.txt', 'ALBUMS 97.09.01.txt', 'ALBUMS 97.10.01.txt', 'ALBUMS 97.11.01.txt', 'ALBUMS 97.12.01.txt'])

for year in [Year.Y1977.value, Year.Y1997.value]:
    review_corpus[year] = remove_header_lines(raw_review_corpus[year], r'^(Reviewed By:|Artist:|Album:|Record Label:).*$')
    review_tokens[year] = tokeniser.tokenize(review_corpus[year])
    scrubbed_tokens[year] = do_scrubbing(review_tokens[year])
    

Initial token count: 38687
Final token count after scrubbing: 17600
Initial token count: 46606
Final token count after scrubbing: 21639


Before looking at frequencies, a few simple metrics:

In [29]:
from nltk.text import Text
import pandas as pd
raw_review_tokens = [None, None]
headings = ['Year', 'Token Count', 'Review Count', 'Paragraph Count', 'Paragraphs per Review', 'Tokens per Review', 'TTR']
data = []
for year in [Year.Y1977.value, Year.Y1997.value]:
    token_type_count = len(set(scrubbed_tokens[year]))
    token_count = len(scrubbed_tokens[year])
    TTR = token_type_count / token_count
    paragraphs = review_corpus[year].split('\n')
    num_paragraphs = len(paragraphs)
    tokens_per_paragraph = token_count / num_paragraphs
    raw_review_tokens[year] = tokeniser.tokenize(raw_review_corpus[year])
    num_non_reviews = review_tokens[year].count('reviewed')
    num_reviews = raw_review_tokens[year].count('Reviewed') - num_non_reviews
    paragraphs_per_review = num_paragraphs / num_reviews
    tokens_per_review = token_count / num_reviews
    data.append([year, token_count, num_reviews, num_paragraphs, paragraphs_per_review, tokens_per_review, TTR])

df = pd.DataFrame(data, columns=headings )
print(df)

   Year  Token Count  Review Count  Paragraph Count  Paragraphs per Review  \
0     0        17600            83              592               7.132530   
1     1        21639           195              423               2.169231   

   Tokens per Review       TTR  
0         212.048193  0.297216  
1         110.969231  0.321734  


# End of Material Copied from Assignmet 1.

# Original Code Starts Here!

The most striking observation in my first look at Rip It Up's reviews from 1977 and 1997 was what seemed to be a move away from discussions on the technicalities of music and recording to mentions of genre.

There's a little bigram analysis that seems to reinforce that.

In [30]:
for year in [Year.Y1977.value, Year.Y1997.value]:
    bigrams = list(nltk.bigrams(scrubbed_tokens[year]))
    bigrams_fdist = FreqDist(bigrams)
    print(f"Most common bigrams for year {Year(year).name}:")
    for freq in bigrams_fdist.most_common(10):
        print(freq)


Most common bigrams for year Y1977:
(('rock', 'n'), 25)
(('n', 'roll'), 24)
(('rhythm', 'section'), 13)
(('first', 'album'), 13)
(('song', 'like'), 12)
(('sound', 'like'), 12)
(('side', 'one'), 10)
(('live', 'album'), 10)
(('rock', 'roll'), 10)
(('new', 'album'), 9)
Most common bigrams for year Y1997:
(('n', 'roll'), 27)
(('rock', 'n'), 26)
(('sound', 'like'), 21)
(('record', 'company'), 16)
(('hip', 'hop'), 15)
(('punk', 'rock'), 13)
(('new', 'album'), 10)
(('debut', 'album'), 9)
(('last', 'year'), 8)
(('r', 'b'), 8)


While it's nice that both corpora have renditions of "rock 'n roll" at the top, "rhythm section" gives way to "hip hop" and "punk rock".


In [31]:
for year in [Year.Y1977.value, Year.Y1997.value]:
    trigrams = list(nltk.trigrams(scrubbed_tokens[year]))
    trigrams_fdist = FreqDist(trigrams)
    print(f"Most common trigrams for year {Year(year).name}:")
    for freq in trigrams_fdist.most_common(10):
        print(freq)

Most common trigrams for year Y1977:
(('rock', 'n', 'roll'), 24)
(('average', 'white', 'band'), 5)
(('rhythm', 'n', 'blue'), 4)
(('day', 'dog', 'race'), 3)
(('time', 'love', 'hero'), 3)
(('miracle', 'billy', 'paul'), 3)
(('allman', 'brother', 'band'), 3)
(('ben', 'e', 'king'), 3)
(('walk', 'wild', 'side'), 3)
(('rock', 'roll', 'heart'), 3)
Most common trigrams for year Y1997:
(('rock', 'n', 'roll'), 26)
(('drum', 'n', 'bass'), 5)
(('blue', 'sky', 'mar'), 3)
(('voodoo', 'glow', 'skull'), 3)
(('ocean', 'colour', 'scene'), 3)
(('simon', 'le', 'bon'), 3)
(('yeah', 'yeah', 'yeah'), 3)
(('dance', 'hall', 'crasher'), 3)
(('plastic', 'ono', 'band'), 3)
(('everything', 'touch', 'run'), 3)


Trigrams are a little murkier with band names appearing frequently in both top tens. Fortunately "average white band" is a band name and not a slur. 

In [ ]:
# BORROWED FROM COURSE MATERIAL!
# make a helper function to create dictionaries.

# to grab the resource by url, we'll import requests
# could also use !wget or other URL libraries
import requests

# create a function to read in resource and output a dictionary.
def get_word_rating_resource(url):
  """helper function to get lexical resources
  resources are hosted on github as .txt files in the form of Word\tValue\n
  """
  # read the raw text and split on newlines
  raw = requests.get(url).text.split('\n')

  # split each pair and convert value to rounded float
  # the if statement is there to avoid indexing errors when a row in a resource doesn't have complete data
  raw_list = [(pair.split('\t')[0], round(float(pair.split('\t')[1]), 3)) for pair in raw if len(pair.split('\t')) == 2]

  # create a dictionary and return it
  return dict(raw_list)

In [62]:
import os
import re
from collections import Counter
# import pandas as pd
# import numpy as np

# Path to your folder containing Hansard .txt files
folder_path = 'hansard'

def get_hansard_frequencies(folder):
    word_counts = Counter()
    total_words = 0
    
    for filename in os.listdir(folder):
        if filename.endswith('.txt'):
            with open(os.path.join(folder, filename), 'r', encoding='utf-8') as f:
                # Basic cleaning: lowercase and keep only alphanumeric characters
                text = f.read().lower()
                words = re.findall(r'\b\w+\b', text)
                
                word_counts.update(words)
                total_words += len(words)
    
    # Convert to DataFrame
    df = pd.DataFrame(word_counts.items(), columns=['word', 'raw_count'])
    
    # Normalize to Frequency Per Million Words (FPMW)
    df['fpmw'] = (df['raw_count'] / total_words) * 1_000_000
    
    # Apply Log10 transformation
    df['log10_fpmw'] = np.log10(df['fpmw'] + 1)
    
    return df, total_words

# Run the analysis
hansard_df, grand_total = get_hansard_frequencies(folder_path)
print(f"Processed {grand_total} total words from Hansard.")
print(hansard_df.sort_values(by='fpmw', ascending=False).head(10))
hansard_dict = dict(zip(hansard_df['word'], hansard_df['log10_fpmw']))


Processed 8016058 total words from Hansard.
     word  raw_count          fpmw  log10_fpmw
44    the     619416  77271.895987    4.888027
4      of     274796  34280.690085    4.535062
57     to     230936  28809.172788    4.459546
65    and     168409  21008.954776    4.322425
632  that     158279  19745.241364    4.295484
497    in     152088  18972.916613    4.278157
473     a     136013  16967.566851    4.229645
61     is     107188  13371.659736    4.126218
73    for      91932  11468.479894    4.059544
557     i      88543  11045.703512    4.043233


In [64]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rtatman/english-word-frequency")

print("Path to dataset files:", path)

SUBTLEXUS_url = 'https://raw.githubusercontent.com/scskalicky/LING-226-vuw/main/lexical-resources/subtlxus_frequency.txt'
SUBTLEXUS_dict = get_word_rating_resource(SUBTLEXUS_url)

import pandas as pd
import numpy as np


# Read the CSV file
df = pd.read_csv(f"{path}/unigram_freq.csv")

total_count = df['count'].sum()
print(f"Total count: {total_count}")
TOTAL_GOOGLE_CORPUS_WORDS = 100000000000

df['fpmw'] = df['count'] / TOTAL_GOOGLE_CORPUS_WORDS * 1000000
df['log10_fpmw'] = round(np.log10(df['fpmw'] + 1), 3)
print(df.head())

# put word and log10_frequency into a dictionary
google_dict = dict(zip(df['word'], df['log10_fpmw']))


Path to dataset files: C:\Users\OEM\.cache\kagglehub\datasets\rtatman\english-word-frequency\versions\1
Total count: 588124220187
  word        count          fpmw  log10_fpmw
0  the  23135851162  231358.51162       5.364
1   of  13151942776  131519.42776       5.119
2  and  12997637966  129976.37966       5.114
3   to  12136980858  121369.80858       5.084
4    a   9081174698   90811.74698       4.958


In [ ]:
def calculate_metric(text, dictionary):
  # create empty output container
  metric = []

  # tokenize the text (and any other preprocessing you might need)
  tokens = nltk.word_tokenize(text)

  # check if token is in dictionary and append to metric output if so
  for token in tokens:
    if token in dictionary.keys():
      metric.append(dictionary[token])
  return metric

def calculate_metric_tokenised(tokens, dictionary):
  # create empty output container
  metric = []

  # check if token is in dictionary and append to metric output if so
  for token in tokens:
    if token in dictionary.keys():
      metric.append(dictionary[token])
  return metric

In [63]:
# metric = calculate_metric('This is a sample text with several words. This text is for testing.', google_dict)
# print(metric)
# print(sum(metric)/len(metric))
# metric = calculate_metric('This is a sample text with several words. This text is for testing.', SUBTLEXUS_dict)
# print(metric)
# print(sum(metric)/len(metric))

for year in [Year.Y1977.value, Year.Y1997.value]:
    metric = calculate_metric_tokenised(scrubbed_tokens[year], google_dict)
    print(f"Average Google log10 frequency for year {Year(year).name}: {sum(metric)/len(metric):.2f}. Coverage: {len(metric)}/{len(scrubbed_tokens[year])} tokens.")
    metric = calculate_metric_tokenised(scrubbed_tokens[year], SUBTLEXUS_dict)
    print(f"Average SUBTLEXUS log10 frequency for year {Year(year).name}: {sum(metric)/len(metric):.2f}. Coverage: {len(metric)}/{len(scrubbed_tokens[year])} tokens.")
    metric = calculate_metric_tokenised(scrubbed_tokens[year], hansard_dict)
    print(f"Average Hansard log10 frequency for year {Year(year).name}: {sum(metric)/len(metric):.2f}. Coverage: {len(metric)}/{len(scrubbed_tokens[year])} tokens.")

Average Google log10 frequency for year Y1977: 2.46. Coverage: 17231/17600 tokens.
Average SUBTLEXUS log10 frequency for year Y1977: 3.32. Coverage: 14731/17600 tokens.
Average Hansard log10 frequency for year Y1977: 1.57. Coverage: 15328/17600 tokens.
Average Google log10 frequency for year Y1997: 2.37. Coverage: 20914/21639 tokens.
Average SUBTLEXUS log10 frequency for year Y1997: 3.21. Coverage: 18143/21639 tokens.
Average Hansard log10 frequency for year Y1997: 1.47. Coverage: 18300/21639 tokens.


*Corpora created using the [Rip It Up magazine archive at Papers Past](https://paperspast.natlib.govt.nz/periodicals/rip-it-up).*
    
*This archive is licensed for non-commercial use under a Creative Commons Attribution Non-Commercial Share Alike 3.0 (CC BY-NC-SA 3.0) licence. Rip it Up is not available for commercial use without the consent of Propeller Lamont Ltd.*